# Exercise 3.1: Ingestion Patterns and Concurrency

In this exercise, you'll learn about different write patterns in Apache Iceberg using the NYC Yellow Taxi dataset:
- **Batch vs Streaming**: Understand different ingestion patterns
- **Copy-on-Write vs Merge-on-Read**: Compare performance tradeoffs
- **Row-Level Operations**: Updates, deletes, and merges
- **Concurrency**: Handle concurrent writes and conflicts

## Learning Objectives
- Implement batch and streaming-style writes
- Compare COW and MOR strategies
- Use MERGE operations for upserts
- Understand and resolve write conflicts

⚠️ **IMPORTANT**: If you see `ConnectionRefusedError: [Errno 111] Connection refused` when initializing the Spark Session, the Docker setup is most likely running out of RAM and shutting down the Spark JVM. Try shutting down other running notebook kernels — go to the **Running Terminals and Kernels** section in the left sidebar — or increase the amount of RAM allocated to Docker.

## Initialize Spark Session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time
from datetime import datetime, timedelta
import random

spark = SparkSession.builder \
    .appName("IngestionPatterns") \
    .getOrCreate()

print(f"Spark {spark.version} initialized!")

## Create Namespace

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS polaris.ingestion")
print("Namespace 'ingestion' created!")

## Download NYC Taxi Data

We'll use NYC Yellow Taxi trip data for **June through August 2023**.

In [ ]:
import boto3
from botocore.client import Config
import urllib.request
import os

s3_client = boto3.client(
    's3',
    endpoint_url='http://minio:9000',
    aws_access_key_id='admin',
    aws_secret_access_key='password',
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

base_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-{:02d}.parquet"
bucket = "warehouse"

for month in [6, 7, 8]:
    filename = f"yellow_tripdata_2023-{month:02d}.parquet"
    key = f"raw/{filename}"
    try:
        s3_client.head_object(Bucket=bucket, Key=key)
        print(f"{filename} already in MinIO, skipping download")
    except:
        local_path = f"/tmp/{filename}"
        print(f"Downloading {filename} (~45MB)...")
        urllib.request.urlretrieve(base_url.format(month), local_path)
        s3_client.upload_file(local_path, bucket, key)
        os.remove(local_path)
        print(f"  Uploaded to s3a://{bucket}/{key}")

print("\nAll taxi data ready in MinIO!")

## Part 1: Batch vs Streaming Ingestion Patterns

### Batch Ingestion Pattern

In traditional databases, the engine manages write frequency transparently. You can do thousands of small inserts and the database handles compaction internally. In Iceberg, every commit creates a new snapshot and potentially new data files, so the frequency and size of writes directly affect metadata accumulation and file layout. This makes the distinction between batch (large, infrequent writes) and streaming (small, frequent writes) important.

Large infrequent writes. Typical for daily ETL jobs.

In [ ]:
spark.sql("DROP TABLE IF EXISTS polaris.ingestion.taxi_trips")

spark.sql("""
    CREATE TABLE polaris.ingestion.taxi_trips
    USING iceberg
    PARTITIONED BY (days(tpep_pickup_datetime))
    AS SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-06.parquet` WHERE 1=0
""")

print("Taxi trips table created!")

In [ ]:
june_df = spark.read.parquet("s3a://warehouse/raw/yellow_tripdata_2023-06.parquet")

print(f"Batch loading {june_df.count():,} June trips...")

start = time.time()
june_df.writeTo("polaris.ingestion.taxi_trips").append()
batch_time = time.time() - start

print(f"Batch write completed in {batch_time:.2f} seconds")

In [ ]:
print("Snapshots after batch write:")
spark.sql("""
    SELECT committed_at, operation, summary['added-records'] as records_added
    FROM polaris.ingestion.taxi_trips.snapshots
    ORDER BY committed_at
""").show(truncate=False)

### Streaming Helper

> **Note:** You do not need to understand the implementation of this helper. It uses Spark Structured Streaming to read data from a staging directory and stream it into an Iceberg table in micro-batches. Adjust the parameters to control the streaming behavior.

In [ ]:
import shutil, os

def run_streaming_ingest(source_path, table_name,
                         num_source_files=10, max_files_per_trigger=2):
    """Stream parquet data into an Iceberg table using Spark Structured Streaming.

    Parameters:
        source_path:           S3 path to a source parquet file
        table_name:            Target Iceberg table (must already exist)
        num_source_files:      Split source into this many small files for streaming
        max_files_per_trigger: Files read per micro-batch (fewer = more snapshots)

    Returns the number of new snapshots created.
    """
    staging = "s3a://warehouse/staging/streaming_input"
    checkpoint = "/tmp/iceberg_streaming_checkpoint"

    if os.path.exists(checkpoint):
        shutil.rmtree(checkpoint)

    df = spark.read.parquet(source_path)
    schema = df.schema
    total_rows = df.count()

    print(f"Staging {total_rows:,} rows as {num_source_files} files...")
    df.repartition(num_source_files).write.mode("overwrite").parquet(staging)

    before = spark.sql(
        f"SELECT COUNT(*) FROM {table_name}.snapshots"
    ).collect()[0][0]

    print(f"Streaming with maxFilesPerTrigger={max_files_per_trigger} "
          f"(expect ~{num_source_files // max_files_per_trigger} micro-batches)...")
    start = time.time()

    query = (spark.readStream
             .schema(schema)
             .option("maxFilesPerTrigger", max_files_per_trigger)
             .parquet(staging)
             .writeStream
             .format("iceberg")
             .outputMode("append")
             .option("checkpointLocation", checkpoint)
             .toTable(table_name))

    query.processAllAvailable()
    query.stop()

    elapsed = time.time() - start
    after = spark.sql(
        f"SELECT COUNT(*) FROM {table_name}.snapshots"
    ).collect()[0][0]
    new_snapshots = after - before

    print(f"Done in {elapsed:.2f}s, created {new_snapshots} new snapshots")
    return new_snapshots

print("Streaming helper loaded!")

### Streaming Ingestion Pattern

Small frequent writes. Typical for real-time streaming.

In [ ]:
streaming_snapshots = run_streaming_ingest(
    source_path="s3a://warehouse/raw/yellow_tripdata_2023-07.parquet",
    table_name="polaris.ingestion.taxi_trips",
    num_source_files=10,
    max_files_per_trigger=2
)

In [ ]:
snapshot_count = spark.sql("""
    SELECT COUNT(*) as count FROM polaris.ingestion.taxi_trips.snapshots
""").collect()[0]['count']

print(f"Total snapshots: {snapshot_count}")

Streaming creates many small snapshots compared to batch's single large snapshot. Each snapshot adds metadata overhead: more manifests, more small files to manage. Unlike traditional databases that merge small writes in the background, Iceberg commits are immutable. Each one writes new files to object storage. The maintenance procedures in E3.2 fill this gap: compacting small files and cleaning up old metadata. The tradeoff is: more frequent commits mean fresher data but more maintenance.

### Try It: Adjust Streaming Parameters

Use `run_streaming_ingest` to stream August data with different settings. Try changing `num_source_files` (how many files the source is split into) and `max_files_per_trigger` (how many files are read per micro-batch). More micro-batches means more snapshots and more small files, the classic streaming trade-off.

In [ ]:
# Try with many small micro-batches (more snapshots, more files):
# run_streaming_ingest(
#     source_path="s3a://warehouse/raw/yellow_tripdata_2023-08.parquet",
#     table_name="polaris.ingestion.taxi_trips",
#     num_source_files=20,
#     max_files_per_trigger=1
# )

# Or fewer, larger micro-batches (fewer snapshots):
# run_streaming_ingest(
#     source_path="s3a://warehouse/raw/yellow_tripdata_2023-08.parquet",
#     table_name="polaris.ingestion.taxi_trips",
#     num_source_files=20,
#     max_files_per_trigger=10
# )

# Compare the results
# spark.sql("SELECT COUNT(*) as snapshots FROM polaris.ingestion.taxi_trips.snapshots").show()
# spark.sql("SELECT COUNT(*) as files FROM polaris.ingestion.taxi_trips.files").show()

## Part 2: Copy-on-Write vs Merge-on-Read

Unlike traditional databases that modify rows in place, Iceberg stores data in immutable files on object storage. To "update" a row, Iceberg must either rewrite the entire file containing that row (COW), or log the change separately (MOR).

**COW (Copy-on-Write)** rewrites entire data files on updates. Slower writes, faster reads.
**MOR (Merge-on-Read)** writes small delete files. Faster writes, slower reads (merge overhead at read time).

> **Note on benchmarks:** The timings in this section are meant to illustrate the *relative* tradeoffs between COW and MOR. Absolute numbers will vary based on your local environment (CPU, memory, disk speed, Docker resource allocation). In production with distributed clusters and larger datasets, the differences are typically more pronounced.

### Create COW and MOR Tables

In [ ]:
for mode_suffix, mode_value in [('cow', 'copy-on-write'), ('mor', 'merge-on-read')]:
    spark.sql(f"DROP TABLE IF EXISTS polaris.ingestion.taxi_{mode_suffix}")
    spark.sql(f"""
        CREATE TABLE polaris.ingestion.taxi_{mode_suffix}
        USING iceberg
        TBLPROPERTIES (
            'write.update.mode' = '{mode_value}',
            'write.delete.mode' = '{mode_value}',
            'write.merge.mode' = '{mode_value}'
        )
        AS SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-06.parquet`
    """)

row_count = spark.sql("SELECT COUNT(*) FROM polaris.ingestion.taxi_cow").collect()[0][0]
size_mb = spark.sql("""
    SELECT ROUND(SUM(file_size_in_bytes) / 1024 / 1024, 1)
    FROM polaris.ingestion.taxi_cow.files
""").collect()[0][0]
print(f"COW and MOR tables created with {row_count:,} rows each (~{size_mb} MB)")

### Compare UPDATE Performance

In [ ]:
start = time.time()
spark.sql("""
    UPDATE polaris.ingestion.taxi_cow
    SET fare_amount = fare_amount * 1.10
    WHERE trip_distance > 10
""")
cow_time = time.time() - start
print(f"COW UPDATE took {cow_time:.3f} seconds")

In [ ]:
start = time.time()
spark.sql("""
    UPDATE polaris.ingestion.taxi_mor
    SET fare_amount = fare_amount * 1.10
    WHERE trip_distance > 10
""")
mor_time = time.time() - start
print(f"MOR UPDATE took {mor_time:.3f} seconds")

if mor_time < cow_time:
    print(f"\nMOR was {cow_time/mor_time:.1f}x faster for writes")
else:
    print(f"\nCOW was {mor_time/cow_time:.1f}x faster for writes (small table effect)")

### Examine File Structure

In [ ]:
print("COW data files:")
spark.sql("""
    SELECT SUBSTRING(file_path, LENGTH(file_path) - LOCATE('/', REVERSE(file_path)) + 2) as filename,
           file_size_in_bytes, record_count
    FROM polaris.ingestion.taxi_cow.files
""").show(truncate=False)

In [ ]:
print("MOR data and delete files:")
spark.sql("""
    SELECT SUBSTRING(file_path, LENGTH(file_path) - LOCATE('/', REVERSE(file_path)) + 2) as filename,
           file_size_in_bytes, record_count
    FROM polaris.ingestion.taxi_mor.files
""").show(truncate=False)

**COW** has only a **single data file**. The update rewrote the entire file, producing one new file that contains both the modified rows (with the 10% fare increase) and the unchanged rows. The original file was replaced.

**MOR** kept the original data file and wrote a small **delete file** alongside it. A delete file contains the positions (row numbers) of rows that have been changed or removed. At read time, Spark skips those positions in the original data file and reads the replacement rows from the new data file, merging them together.

### Compare READ Performance

To measure **read** overhead without bottlenecking on Python serialization, we sum a column. This forces Spark to read every value from every data file, but only a single number comes back to the driver.

In [ ]:
scan_query = "SELECT SUM(fare_amount) FROM {}"

start = time.time()
spark.sql(scan_query.format("polaris.ingestion.taxi_cow")).collect()
cow_read = time.time() - start

start = time.time()
spark.sql(scan_query.format("polaris.ingestion.taxi_mor")).collect()
mor_read = time.time() - start

print(f"COW READ: {cow_read:.3f}s")
print(f"MOR READ: {mor_read:.3f}s")
if cow_read < mor_read:
    print(f"\nCOW reads were {mor_read/cow_read:.1f}x faster (no delete-file merge overhead)")
else:
    print(f"\nResults similar at this scale")

### Try It: How Does Selectivity Affect COW vs MOR?

The update above touched rows where `trip_distance > 10`, a small fraction of the data. What happens when you update a larger or smaller portion?

Try changing `update_pct` below and re-running. Some examples to try:

| `update_pct` | Approximate Filter | Rows Affected |
|---|---|---|
| `1` | Longest 1% of trips | ~33K rows |
| `10` | Longer-than-average trips | ~330K rows |
| `50` | Half the table | ~1.6M rows |
| `90` | Nearly everything | ~3M rows |

**Prediction:** MOR should shine at low percentages (small delete files). As the percentage grows, COW's single-file rewrite becomes competitive because MOR accumulates large delete files that slow reads.

In [ ]:
def benchmark_cow_vs_mor(update_pct):
    """Recreate COW and MOR tables, update a percentage of rows, and compare write/read performance."""
    for mode_suffix, mode_value in [('cow', 'copy-on-write'), ('mor', 'merge-on-read')]:
        spark.sql(f"DROP TABLE IF EXISTS polaris.ingestion.taxi_{mode_suffix}")
        spark.sql(f"""
            CREATE TABLE polaris.ingestion.taxi_{mode_suffix}
            USING iceberg
            TBLPROPERTIES (
                'write.update.mode' = '{mode_value}',
                'write.delete.mode' = '{mode_value}',
                'write.merge.mode' = '{mode_value}'
            )
            AS SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-06.parquet`
        """)

    total = spark.sql("SELECT COUNT(*) FROM polaris.ingestion.taxi_cow").collect()[0][0]

    threshold = spark.sql(f"""
        SELECT PERCENTILE_APPROX(trip_distance, {(100 - update_pct) / 100.0})
        FROM polaris.ingestion.taxi_cow
    """).collect()[0][0]

    affected = spark.sql(f"""
        SELECT COUNT(*) FROM polaris.ingestion.taxi_cow WHERE trip_distance >= {threshold}
    """).collect()[0][0]
    print(f"Updating rows with trip_distance >= {threshold:.2f}")
    print(f"Affects {affected:,} / {total:,} rows ({100*affected/total:.1f}%)")
    print("=" * 60)

    start = time.time()
    spark.sql(f"UPDATE polaris.ingestion.taxi_cow SET fare_amount = fare_amount * 1.10 WHERE trip_distance >= {threshold}")
    cow_write = time.time() - start

    start = time.time()
    spark.sql(f"UPDATE polaris.ingestion.taxi_mor SET fare_amount = fare_amount * 1.10 WHERE trip_distance >= {threshold}")
    mor_write = time.time() - start

    scan_query = "SELECT SUM(fare_amount) FROM {}"
    start = time.time()
    spark.sql(scan_query.format("polaris.ingestion.taxi_cow")).collect()
    cow_read = time.time() - start

    start = time.time()
    spark.sql(scan_query.format("polaris.ingestion.taxi_mor")).collect()
    mor_read = time.time() - start

    print(f"\n{'Metric':<15} {'COW':>10} {'MOR':>10} {'Winner':>10}")
    print("-" * 50)
    print(f"{'Write':.<15} {cow_write:>9.3f}s {mor_write:>9.3f}s {'MOR' if mor_write < cow_write else 'COW':>10}")
    print(f"{'Read':.<15} {cow_read:>9.3f}s {mor_read:>9.3f}s {'COW' if cow_read < mor_read else 'MOR':>10}")

print("benchmark_cow_vs_mor() helper ready")

In [ ]:
# benchmark_cow_vs_mor(update_pct=???)  # <-- Try: 1, 10, 50, 90

## Part 3: Row-Level Operations

Iceberg supports several SQL operations that trigger **row-level** changes, meaning Iceberg must identify and modify individual rows within data files, not just append new files. Depending on the table's write mode (COW or MOR from Part 2), these operations either rewrite entire data files or produce delete files.

| Operation | What It Does | Example |
|---|---|---|
| `DELETE` | Remove rows matching a predicate | `DELETE FROM t WHERE fare < 0` |
| `UPDATE` | Modify column values in matching rows | `UPDATE t SET fare = fare * 1.1 WHERE ...` |
| `MERGE INTO` | Upsert: update existing rows, insert new ones | `MERGE INTO t USING src ON ... WHEN MATCHED THEN UPDATE ... WHEN NOT MATCHED THEN INSERT ...` |

All three share the same underlying mechanism. Iceberg scans for affected files using column metrics, applies the change via COW or MOR, and atomically commits a new snapshot. Let's walk through each one.

### DELETE Operations

In [ ]:
spark.sql("DROP TABLE IF EXISTS polaris.ingestion.taxi_partitioned")

spark.sql("""
    CREATE TABLE polaris.ingestion.taxi_partitioned
    USING iceberg
    PARTITIONED BY (days(tpep_pickup_datetime))
    AS SELECT VendorID, tpep_pickup_datetime, passenger_count, trip_distance,
              fare_amount, tip_amount, total_amount
       FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-06.parquet`
""")

count = spark.sql("SELECT COUNT(*) FROM polaris.ingestion.taxi_partitioned").collect()[0][0]
print(f"Partitioned table created with {count:,} trips")

In [ ]:
bad_fares = spark.sql("""
    SELECT COUNT(*) FROM polaris.ingestion.taxi_partitioned WHERE fare_amount <= 0
""").collect()[0][0]

start = time.time()
spark.sql("DELETE FROM polaris.ingestion.taxi_partitioned WHERE fare_amount <= 0")
delete_time = time.time() - start

remaining = spark.sql("SELECT COUNT(*) FROM polaris.ingestion.taxi_partitioned").collect()[0][0]
print(f"Deleted {bad_fares:,} bad-fare trips in {delete_time:.3f} seconds")
print(f"Remaining: {remaining:,} trips")

### Metadata-Only Deletes

When every row in a data file matches the delete predicate, Iceberg can remove the entire file from the metadata **without reading or rewriting any data**. This is extremely fast. It works because Iceberg stores per-column **min/max statistics** in each file's metadata. If the delete predicate's range completely covers a file's value range, the file can be dropped. Metadata deletes happen automatically when possible.

Let's set this up with a table where files have clean, non-overlapping value ranges.

In [ ]:
spark.sql("DROP TABLE IF EXISTS polaris.ingestion.taxi_meta_delete")

spark.sql("""
    CREATE TABLE polaris.ingestion.taxi_meta_delete
    USING iceberg
    PARTITIONED BY (days(tpep_pickup_datetime))
    AS SELECT VendorID, tpep_pickup_datetime, passenger_count,
              trip_distance, fare_amount, tip_amount, total_amount
       FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-06.parquet`
""")

count = spark.sql("SELECT COUNT(*) FROM polaris.ingestion.taxi_meta_delete").collect()[0][0]
partitions = spark.sql("""
    SELECT COUNT(DISTINCT DATE(tpep_pickup_datetime)) FROM polaris.ingestion.taxi_meta_delete
""").collect()[0][0]
print(f"Table created: {count:,} trips across {partitions} day-partitions (one file per day)")

Each day-partition has its own data file. Let's look at the **readable_metrics** for these files, specifically the `tpep_pickup_datetime` min/max bounds. This is what Iceberg uses to decide whether a file can be dropped without reading it.

In [ ]:
print("File-level readable metrics for tpep_pickup_datetime:\n")

spark.sql("""
    SELECT
        SUBSTRING(file_path, LENGTH(file_path) - LOCATE('/', REVERSE(file_path)) + 2) AS file_name,
        record_count,
        readable_metrics.tpep_pickup_datetime.lower_bound AS pickup_min,
        readable_metrics.tpep_pickup_datetime.upper_bound AS pickup_max
    FROM polaris.ingestion.taxi_meta_delete.files
    ORDER BY pickup_min
    LIMIT 10
""").show(truncate=False)

Each file's `pickup_min` and `pickup_max` fall entirely within a single day. If we issue a `DELETE WHERE tpep_pickup_datetime >= '2023-06-01' AND tpep_pickup_datetime < '2023-06-02'`, Iceberg checks the file metrics and finds that the June 1 file's entire range is covered by the predicate. It can drop that file from the metadata with **no data scan, no rewrite**.

In [ ]:
rows_before = spark.sql("SELECT COUNT(*) FROM polaris.ingestion.taxi_meta_delete").collect()[0][0]
june1_rows = spark.sql("""
    SELECT COUNT(*) FROM polaris.ingestion.taxi_meta_delete
    WHERE tpep_pickup_datetime >= '2023-06-01' AND tpep_pickup_datetime < '2023-06-02'
""").collect()[0][0]

start = time.time()
spark.sql("""
    DELETE FROM polaris.ingestion.taxi_meta_delete
    WHERE tpep_pickup_datetime >= '2023-06-01' AND tpep_pickup_datetime < '2023-06-02'
""")
delete_time = time.time() - start

rows_after = spark.sql("SELECT COUNT(*) FROM polaris.ingestion.taxi_meta_delete").collect()[0][0]
print(f"Deleted {june1_rows:,} rows (entire June 1 partition) in {delete_time:.3f}s")
print(f"Rows: {rows_before:,} → {rows_after:,}")

In [ ]:
print("Snapshot summary for the delete operation:\n")

spark.sql("""
    SELECT operation,
           summary['deleted-data-files'] AS files_removed,
           summary['added-data-files'] AS files_added,
           summary['deleted-records'] AS records_deleted,
           summary['added-records'] AS records_added,
           summary['total-records'] AS total_remaining
    FROM polaris.ingestion.taxi_meta_delete.snapshots
    ORDER BY committed_at DESC
    LIMIT 1
""").show(truncate=False)

Notice: **files_removed = 1**, **files_added = 0**. Iceberg didn't read or rewrite any data. It simply removed one file reference from the metadata. This is what makes partition-aligned deletes so fast in Iceberg. Compare this with a traditional database, which would need to scan and mark individual rows for deletion.

### MERGE Operations (Upserts)

In [ ]:
spark.sql("DROP TABLE IF EXISTS polaris.ingestion.taxi_corrections")

spark.sql("""
    CREATE TABLE polaris.ingestion.taxi_corrections
    USING iceberg
    AS SELECT VendorID, tpep_pickup_datetime, passenger_count,
              trip_distance, fare_amount, tip_amount, total_amount
       FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-06.parquet`
       LIMIT 1000
""")

print(f"Corrections table created with 1,000 trips")

In [ ]:
corrections = spark.sql("""
    SELECT VendorID, tpep_pickup_datetime, passenger_count,
           trip_distance, fare_amount * 1.05 as fare_amount,
           tip_amount, total_amount * 1.05 as total_amount
    FROM polaris.ingestion.taxi_corrections
    LIMIT 100
""")
corrections.createOrReplaceTempView("fare_corrections")

print("Prepared 100 fare corrections (5% increase)")

In [ ]:
spark.sql("""
    MERGE INTO polaris.ingestion.taxi_corrections t
    USING fare_corrections s
    ON t.tpep_pickup_datetime = s.tpep_pickup_datetime
       AND t.VendorID = s.VendorID
       AND t.trip_distance = s.trip_distance
    WHEN MATCHED THEN UPDATE SET
        t.fare_amount = s.fare_amount,
        t.total_amount = s.total_amount
    WHEN NOT MATCHED THEN INSERT *
""")

print("MERGE completed!")

In [ ]:
print("Corrections table after MERGE:")
spark.sql("""
    SELECT COUNT(*) as total_rows,
           ROUND(AVG(fare_amount), 2) as avg_fare,
           ROUND(AVG(total_amount), 2) as avg_total
    FROM polaris.ingestion.taxi_corrections
""").show()

### Try It: Write Your Own MERGE

Create a small corrections dataset (e.g., adjust `tip_amount` for specific trips) and MERGE it into `taxi_corrections`. Try using both `WHEN MATCHED THEN UPDATE` and `WHEN NOT MATCHED THEN INSERT` clauses. Check the snapshot history to see the MERGE operation recorded.

In [ ]:
# Step 1: Create a corrections view. Change the column and factor
# IMPORTANT: The ON clause columns must uniquely identify rows. The combination
# (tpep_pickup_datetime, VendorID, trip_distance) may have duplicates in the
# full dataset, causing a MERGE_CARDINALITY_VIOLATION. With LIMIT 50 from
# an already-small table this is unlikely, but if you hit it, add more columns
# to the ON clause (e.g., fare_amount, tip_amount) to make matches unique.
# spark.sql("""
#     SELECT VendorID, tpep_pickup_datetime, passenger_count,
#            trip_distance, fare_amount, tip_amount * ??? as tip_amount, total_amount
#     FROM polaris.ingestion.taxi_corrections
#     LIMIT 50
# """).createOrReplaceTempView("my_corrections")

# Step 2: MERGE your corrections. Fill in the WHEN clauses
# spark.sql("""
#     MERGE INTO polaris.ingestion.taxi_corrections t
#     USING my_corrections s
#     ON t.tpep_pickup_datetime = s.tpep_pickup_datetime
#        AND t.VendorID = s.VendorID
#        AND t.trip_distance = s.trip_distance
#     WHEN MATCHED THEN UPDATE SET t.tip_amount = s.tip_amount
#     WHEN NOT MATCHED THEN INSERT *
# """)

# Step 3: Check snapshot history for your MERGE operation
# spark.sql("SELECT committed_at, operation FROM polaris.ingestion.taxi_corrections.snapshots ORDER BY committed_at").show(truncate=False)

## Part 4: Concurrency and Conflicts

Traditional databases use locks (pessimistic concurrency) to prevent conflicts. Two transactions can update different rows in the same table without blocking each other. Iceberg uses **optimistic concurrency** instead: writers proceed independently and only check for conflicts at commit time.

Conflicts are detected at the **file level** using column-level min/max statistics (the same `readable_metrics` we saw earlier). When a writer commits, Iceberg checks whether the new commit's affected files overlap with files changed by any concurrent commit. This validation is engine and isolation level specific, but generally works by comparing a predicate passed to Iceberg in a "ON" or "WHERE" clause to files in the table. If two queries could only possibly touch different files, there won't be a conflict. 

### Concurrency Helper

To demonstrate real concurrent writes we need to run multiple Spark SQL statements at the same time. Python's `ThreadPoolExecutor` works well here because `spark.sql()` releases the GIL while the JVM does the actual work.

Don't worry about understanding this code since it's not really Iceberg specific and is more about Python/Java internals. 

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def run_concurrent_sqls(statements):
    """Run N named SQL statements concurrently and report results.
    
    Args:
        statements: list of (name, sql_string) tuples
    Returns:
        list of (name, status, error_snippet) tuples
    """
    def _extract_message(e):
        """Pull the meaningful exception line from a PySpark/Py4j stacktrace."""
        msg = str(e)
        for line in msg.split("\n"):
            line = line.strip()
            for marker in ["CommitFailedException", "ValidationException", "CommitStateUnknownException"]:
                if marker in line:
                    idx = line.find(marker)
                    return line[idx:][:200]
        return msg.split("\n")[0][:200]

    def _exec(name, sql):
        try:
            spark.sql(sql)
            return (name, "OK", None)
        except Exception as e:
            msg = str(e)
            snippet = _extract_message(e)
            conflict_markers = ["CommitFailedException", "ValidationException", "conflicting files"]
            if any(m in msg for m in conflict_markers):
                return (name, "CONFLICT", snippet)
            return (name, "ERROR", snippet)

    results = []
    with ThreadPoolExecutor(max_workers=len(statements)) as pool:
        futures = {
            pool.submit(_exec, name, sql): name
            for name, sql in statements
        }
        for future in as_completed(futures):
            results.append(future.result())

    for name, status, err in sorted(results, key=lambda r: r[0]):
        print(f"  {name}: {status}")
        if err:
            print(f"    --> {err}")
    return results

print("run_concurrent_sqls() helper ready")

### Non-Conflicting Writes

Updates whose predicates resolve to different files succeed without conflict. Partitioning is the most common way to achieve this (one file per partition), but any predicate that excludes files via column metrics will work.

In [ ]:
print("Concurrently updating trips on two different days...\n")

start = time.time()
results = run_concurrent_sqls([
    ("Writer A (June 1)", """
        UPDATE polaris.ingestion.taxi_partitioned
        SET fare_amount = fare_amount * 1.02
        WHERE tpep_pickup_datetime >= '2023-06-01' AND tpep_pickup_datetime < '2023-06-02'
    """),
    ("Writer B (June 15)", """
        UPDATE polaris.ingestion.taxi_partitioned
        SET fare_amount = fare_amount * 1.03
        WHERE tpep_pickup_datetime >= '2023-06-15' AND tpep_pickup_datetime < '2023-06-16'
    """),
])
elapsed = time.time() - start
print(f"\nTotal wall-clock time: {elapsed:.1f}s (both ran in parallel)")

Both updates succeeded because their predicates resolved to different files (each day is in its own partition file). Iceberg checked the column metrics and confirmed there was no file overlap between the two writers.

### Conflicting Writes

What happens when two writers target the **same files**? Both read the current snapshot, do their work, and try to commit. The first commit wins; the second detects that the files it read have been replaced and raises a `CommitFailedException`.

In [ ]:
print("Concurrently updating the SAME partition (June 1) from two writers...\n")

results = run_concurrent_sqls([
    ("Writer A (+2%)", """
        UPDATE polaris.ingestion.taxi_partitioned
        SET fare_amount = fare_amount * 1.02
        WHERE tpep_pickup_datetime >= '2023-06-01' AND tpep_pickup_datetime < '2023-06-02'
    """),
    ("Writer B (+3%)", """
        UPDATE polaris.ingestion.taxi_partitioned
        SET fare_amount = fare_amount * 1.03
        WHERE tpep_pickup_datetime >= '2023-06-01' AND tpep_pickup_datetime < '2023-06-02'
    """),
])

One writer succeeded and the other got a `CommitFailedException`, exactly what Iceberg's optimistic concurrency guarantees. The failed writer **did not corrupt data**; it simply needs to re-read the table and retry.

**Best practice:** Include predicates in your `WHERE` and `MERGE ON` clauses that let Iceberg narrow the conflict-detection window using file-level column metrics. Partition columns are the most effective (each partition maps to its own file), but any column with good min/max separation across files can help.

### Commit Retries and Concurrency

When a commit fails, Iceberg can **automatically retry**: it refreshes the base snapshot and attempts the commit again. The number of retries is controlled by `commit.retry.num-retries` (default: **4**).

It's important to understand **what kinds of failures can be retried**. When multiple writers try to update the catalog simultaneously, the second writer gets a `CommitFailedException` (the branch head moved). For **append** operations (INSERT), this is always safe to retry because appends never conflict with each other, they just need a fresh base snapshot. For **update/delete** operations (COW), the conflict is at the data level (`ValidationException`) and cannot be retried automatically because the data files were already written against the old snapshot.

Let's see this in action with concurrent appends. We'll create a table with low retries and flood it with concurrent INSERTs.

In [ ]:
spark.sql("DROP TABLE IF EXISTS polaris.ingestion.taxi_retry_demo")

spark.sql("""
    CREATE TABLE polaris.ingestion.taxi_retry_demo (
        writer_id INT, batch_id INT, value DOUBLE
    )
    USING iceberg
    TBLPROPERTIES ('commit.retry.num-retries' = '1')
""")

print("Table created with commit.retry.num-retries = 1")
print("Each writer gets 1 original attempt + 1 retry = 2 chances total.")
print("Launching 10 concurrent INSERTs...\n")

results = run_concurrent_sqls([
    (f"Writer {chr(65+i)}", f"""
        INSERT INTO polaris.ingestion.taxi_retry_demo VALUES ({i}, 1, {i * 1.1})
    """)
    for i in range(10)
])

ok_count = sum(1 for _, status, _ in results if status == "OK")
conflict_count = sum(1 for _, status, _ in results if status == "CONFLICT")
error_count = sum(1 for _, status, _ in results if status == "ERROR")
print(f"\nResults: {ok_count} succeeded, {conflict_count + error_count} failed")
if conflict_count + error_count > 0:
    print("Some writers exhausted their single retry and couldn't commit.")

With only 1 retry, some INSERTs couldn't land their commit because the catalog branch kept moving out from under them. Notice these are **appends**, not updates. There's no data-level conflict, just a race to update the catalog pointer. More retries would let all writers eventually serialize.

Let's increase `commit.retry.num-retries` to 20 and try again.

In [ ]:
spark.sql("""
    ALTER TABLE polaris.ingestion.taxi_retry_demo
    SET TBLPROPERTIES ('commit.retry.num-retries' = '20')
""")
print("Set commit.retry.num-retries = 20")
print("Re-running 10 concurrent INSERTs...\n")

results = run_concurrent_sqls([
    (f"Writer {chr(65+i)}", f"""
        INSERT INTO polaris.ingestion.taxi_retry_demo VALUES ({i}, 2, {i * 2.2})
    """)
    for i in range(10)
])

ok_count = sum(1 for _, status, _ in results if status == "OK")
conflict_count = sum(1 for _, status, _ in results if status == "CONFLICT")
error_count = sum(1 for _, status, _ in results if status == "ERROR")
print(f"\nResults: {ok_count} succeeded, {conflict_count + error_count} failed")
if conflict_count + error_count == 0:
    print("All writers succeeded with enough retries to serialize all 10 commits.")

total_rows = spark.sql("SELECT COUNT(*) FROM polaris.ingestion.taxi_retry_demo").collect()[0][0]
print(f"Table now has {total_rows} rows total")

With enough retries, all 10 concurrent INSERTs succeeded. Each writer that lost the catalog race simply refreshed its base snapshot and retried the commit.

**Key insight:** `commit.retry.num-retries` helps with **catalog-level races**, i.e. multiple writers trying to advance the branch pointer at the same time. This applies to appends, manifest updates, and other operations that don't have data-level conflicts. For COW updates/deletes to the same files, the failure is a `ValidationException` (data conflict), not a `CommitFailedException` (catalog race), and requires the application to re-execute the entire operation from scratch. The difference: an append's output files are always valid regardless of the base snapshot, so retrying just means committing them against the new base. But a COW update reads and rewrites specific files. If those files have been replaced by another writer, the rewrite is against stale data and must be redone from scratch.

Related properties:
- `commit.retry.min-wait-ms` (default: 100): minimum backoff between retries
- `commit.retry.max-wait-ms` (default: 60,000): maximum backoff
- `commit.retry.total-timeout-ms` (default: 1,800,000): total timeout for all retries

### MERGE and the ON Clause: Scoping Conflict Detection

With `UPDATE`, Iceberg uses the `WHERE` clause to determine which files are affected. With `MERGE`, it uses the **`ON` clause**. If the `ON` clause includes predicates that can exclude files via column metrics (most commonly a partition column, but any column with useful min/max bounds works), Iceberg can scope the merge to just those files, and concurrent merges targeting non-overlapping file sets can both succeed.

But if the file-scoping predicate is buried in the `WHEN MATCHED` clause instead of the `ON` clause, Iceberg cannot use it for conflict detection. This is because Iceberg determines the set of potentially affected files *before* evaluating `WHEN` conditions. It uses only the `ON` clause predicates to select files, then applies `WHEN` row-by-row after files are already selected. Without a file-scoping filter in `ON`, it must assume the merge could touch **any file in the table**, and concurrent merges will conflict even if they logically target different data.

In [ ]:
spark.sql("DROP TABLE IF EXISTS polaris.ingestion.taxi_merge_demo")

spark.sql("""
    CREATE TABLE polaris.ingestion.taxi_merge_demo (
        id INT, day STRING, amount DOUBLE
    )
    USING iceberg
    PARTITIONED BY (day)
""")

spark.sql("""
    INSERT INTO polaris.ingestion.taxi_merge_demo VALUES
    (1, '2023-06-01', 10.0), (2, '2023-06-01', 20.0),
    (3, '2023-06-02', 30.0), (4, '2023-06-02', 40.0)
""")

spark.sql("""
    SELECT id, day, amount * 1.05 AS new_amount
    FROM polaris.ingestion.taxi_merge_demo
""").createOrReplaceTempView("merge_source")

print("Merge demo table and source view ready")
spark.sql("SELECT * FROM polaris.ingestion.taxi_merge_demo ORDER BY id").show()

**Case 1: File-scoping predicate in the ON clause.** The date range predicate lets Iceberg use column metrics to scope each merge to its own set of files, so two merges targeting different days should both succeed.

In [ ]:
print("Case 1: File-scoping predicate IN the ON clause\n")

results = run_concurrent_sqls([
    ("Merge A (June 1)", """
        MERGE INTO polaris.ingestion.taxi_merge_demo t
        USING merge_source s
        ON t.id = s.id AND t.day = '2023-06-01'
        WHEN MATCHED THEN UPDATE SET t.amount = s.new_amount
    """),
    ("Merge B (June 2)", """
        MERGE INTO polaris.ingestion.taxi_merge_demo t
        USING merge_source s
        ON t.id = s.id AND t.day = '2023-06-02'
        WHEN MATCHED THEN UPDATE SET t.amount = s.new_amount
    """),
])

Both succeeded. Iceberg used the date predicate in the ON clause to check file-level column metrics and confirmed the two merges affected non-overlapping files.

**Case 2: Predicate only in WHEN MATCHED.** The `ON` clause has no file-scoping filter. Iceberg can only use the ON clause for conflict detection, so it must treat the merge as potentially touching every file in the table. Even though the two merges logically target different days, Iceberg can't prove they don't overlap.

In [ ]:
print("Case 2: No file-scoping predicate in the ON clause (only in WHEN MATCHED)\n")

results = run_concurrent_sqls([
    ("Merge A (June 1)", """
        MERGE INTO polaris.ingestion.taxi_merge_demo t
        USING merge_source s
        ON t.id = s.id
        WHEN MATCHED AND t.day = '2023-06-01'
        THEN UPDATE SET t.amount = s.new_amount
    """),
    ("Merge B (June 2)", """
        MERGE INTO polaris.ingestion.taxi_merge_demo t
        USING merge_source s
        ON t.id = s.id
        WHEN MATCHED AND t.day = '2023-06-02'
        THEN UPDATE SET t.amount = s.new_amount
    """),
])

One merge conflicted, even though both targeted different days. Without a file-scoping predicate in the `ON` clause, Iceberg's conflict detection treated both merges as potentially overlapping across every file in the table.

**Takeaway:** Always include file-scoping predicates in the `ON` clause of your `MERGE` statements, not just in the `WHEN` conditions. Partition columns are the most reliable choice (each partition has its own file), but any column with good min/max separation across files will help Iceberg narrow the conflict window. The key insight is that this works because of the same **file-level column metrics** we saw in the metadata-only delete section: Iceberg uses min/max bounds to determine which files a predicate could affect.

### Try It: Use Column Metrics to Find Non-Conflicting Ranges

Partitioning isn't the only way to avoid conflicts. Any predicate that maps exclusively to a subset of files (via column min/max stats) works. To make this concrete, here's an **unpartitioned** table sorted by `fare_amount`. Because the data is sorted, each file has a distinct fare range with no overlap.

**Your challenge:**
1. Inspect the `readable_metrics` to see each file's `fare_amount` min/max bounds
2. Pick two ranges that each fall entirely within a **single, different** file
3. Run concurrent updates with those ranges (they should both succeed)
4. Then try two ranges that overlap the **same** file (one should conflict)

In [ ]:
from pyspark.sql.functions import col

spark.sql("DROP TABLE IF EXISTS polaris.ingestion.taxi_sorted")

df = spark.read.parquet("s3a://warehouse/raw/yellow_tripdata_2023-06.parquet") \
    .select("VendorID", "tpep_pickup_datetime", "passenger_count",
            "trip_distance", "fare_amount", "tip_amount", "total_amount") \
    .repartitionByRange(5, col("fare_amount")) \
    .sortWithinPartitions("fare_amount")

df.writeTo("polaris.ingestion.taxi_sorted") \
    .using("iceberg") \
    .create()

print("Unpartitioned table created, sorted by fare_amount into 5 files")
print("Each file has a distinct, non-overlapping fare_amount range.\n")

spark.sql("""
    SELECT
        SUBSTRING(file_path, LENGTH(file_path) - LOCATE('/', REVERSE(file_path)) + 2) AS file,
        record_count,
        readable_metrics.fare_amount.lower_bound AS fare_min,
        readable_metrics.fare_amount.upper_bound AS fare_max
    FROM polaris.ingestion.taxi_sorted.files
    ORDER BY fare_min
""").show(truncate=False)

In [ ]:
# Step 1: Pick two fare ranges that each fall within a DIFFERENT file
#         (use the min/max values from the metrics above)
# range_a = "fare_amount >= ??? AND fare_amount < ???"
# range_b = "fare_amount >= ??? AND fare_amount < ???"

# Step 2: Concurrent updates to different files (should both succeed)
# run_concurrent_sqls([
#     ("Range A", f"UPDATE polaris.ingestion.taxi_sorted SET tip_amount = tip_amount + 0.01 WHERE {range_a}"),
#     ("Range B", f"UPDATE polaris.ingestion.taxi_sorted SET tip_amount = tip_amount + 0.01 WHERE {range_b}"),
# ])

# Step 3: Now try two ranges that overlap the SAME file (one should conflict)
# overlapping = "fare_amount >= ??? AND fare_amount < ???"
# run_concurrent_sqls([
#     ("Writer A", f"UPDATE polaris.ingestion.taxi_sorted SET tip_amount = tip_amount + 0.01 WHERE {overlapping}"),
#     ("Writer B", f"UPDATE polaris.ingestion.taxi_sorted SET tip_amount = tip_amount + 0.02 WHERE {overlapping}"),
# ])

## Cleanup

In [ ]:
# Optional: Drop tables
# for table in ['taxi_trips', 'taxi_cow', 'taxi_mor', 'taxi_partitioned',
#               'taxi_corrections', 'taxi_meta_delete', 'taxi_merge_demo',
#               'taxi_retry_demo', 'taxi_sorted']:
#     spark.sql(f"DROP TABLE IF EXISTS polaris.ingestion.{table}")
# print("Tables dropped!")